# Loan-Level Outcome Construction

## Purpose

This notebook transforms monthly Freddie Mac performance records into
one analytical outcome record per loan.

The primary modeling outcome is whether a loan reaches 90 or more days
delinquent within its first 24 months of observed performance.

Additional 12-month and 36-month outcomes will be retained for vintage,
sensitivity, and survival-oriented analysis.

## Primary Outcome Definition

A loan receives:

- `default_24m = 1` if it records delinquency status `03` or higher,
  or enters REO status (`RA`), during the first 24 monthly periods
  measured from the original first-payment date (month indexes 0
  through 23);
- `default_24m = 0` if it has a complete 24-month observation window
  without meeting the default condition; and
- an ineligible or censored designation if a complete outcome window
  cannot be established.

Monthly records will be processed one vintage at a time in 100,000-row
chunks to remain within the computer's memory constraints.

## Control Considerations

The construction process will:

1. use only validated performance fields;
2. maintain one final outcome record per loan;
3. independently reconcile derived outcomes;
4. distinguish defaults from incomplete observation windows;
5. prevent post-outcome information from entering predictor variables;
   and
6. document all assumptions, exclusions, and reconciliation results.

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd


# Locate the project root whether Jupyter starts from the
# repository folder or the notebooks folder.
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_data_directory = project_root / "data" / "raw"
processed_data_directory = (
    project_root / "data" / "processed"
)

processed_data_directory.mkdir(
    parents=True,
    exist_ok=True,
)

origination_path = (
    processed_data_directory
    / "originations_clean.parquet"
)

vintages = [2006, 2015, 2016, 2017]


# Locate each performance file without assuming its subfolder.
performance_files = {}

for year in vintages:
    matches = list(
        raw_data_directory.rglob(
            f"sample_perf_{year}.txt"
        )
    )

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one performance file for {year}, "
            f"but found {len(matches)}."
        )

    performance_files[year] = matches[0]


# Official 35-field monthly performance-file schema.
performance_columns = [
    "loan_identifier",
    "monthly_reporting_period",
    "current_actual_upb",
    "current_loan_delinquency_status",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "underwriting_defect_and_major_servicing_defect_settlement_date",
    "modification_flag",
    "zero_balance_code",
    "zero_balance_effective_date",
    "current_interest_rate",
    "current_non_interest_bearing_upb",
    "due_date_of_last_paid_installment",
    "mi_recoveries",
    "net_sales_proceeds",
    "non_mi_recoveries",
    "total_expenses",
    "legal_costs",
    "maintenance_and_preservation_costs",
    "taxes_and_insurance",
    "miscellaneous_expenses",
    "actual_loss",
    "cumulative_modification_costs",
    "interest_rate_step_indicator",
    "payment_deferral_flag",
    "estimated_ltv",
    "zero_balance_removal_upb",
    "delinquent_accrued_interest",
    "delinquency_due_to_disaster",
    "borrower_assistance_plan",
    "current_period_modification_costs",
    "current_interest_bearing_upb",
    "mortgage_insurance_cancellation_indicator",
    "servicer_name",
    "bankruptcy_cramdown_costs",
]


setup_check = pd.DataFrame(
    [
        {
            "vintage": year,
            "file_name": performance_files[year].name,
            "file_exists": performance_files[year].exists(),
            "file_size_mb": round(
                performance_files[year].stat().st_size
                / (1024 ** 2),
                2,
            ),
        }
        for year in vintages
    ]
)

print("Project root:", project_root)
print("Origination file exists:", origination_path.exists())
print("Performance schema fields:", len(performance_columns))

display(setup_check)

Project root: c:\GitHub Projects\freddie-mac-credit-risk
Origination file exists: True
Performance schema fields: 35


,vintage,file_name,file_exists,file_size_mb
0,2006,sample_perf_2006.txt,True,340.70
1,2015,sample_perf_2015.txt,True,355.59
2,2016,sample_perf_2016.txt,True,349.37
3,2017,sample_perf_2017.txt,True,292.56


In [2]:
origination_reference = pd.read_parquet(
    origination_path,
    columns=[
        "loan_identifier",
        "vintage",
        "first_payment_date",
    ],
)

origination_reference["vintage"] = pd.to_numeric(
    origination_reference["vintage"],
    errors="coerce",
).astype("Int64")

origination_reference["first_payment_date"] = (
    pd.to_datetime(
        origination_reference["first_payment_date"],
        errors="coerce",
    )
)

origination_reference_check = pd.Series(
    {
        "origination_rows": len(
            origination_reference
        ),
        "unique_loan_ids": (
            origination_reference[
                "loan_identifier"
            ].nunique()
        ),
        "duplicate_loan_ids": (
            origination_reference[
                "loan_identifier"
            ].duplicated().sum()
        ),
        "missing_first_payment_dates": (
            origination_reference[
                "first_payment_date"
            ].isna().sum()
        ),
    },
    name="result",
)

loans_by_vintage = (
    origination_reference
    .groupby("vintage", dropna=False)
    .size()
    .rename("loan_count")
    .reset_index()
)

display(origination_reference_check)
display(loans_by_vintage)

origination_rows               200000
unique_loan_ids                200000
duplicate_loan_ids                  0
missing_first_payment_dates         0
Name: result, dtype: int64

,vintage,loan_count
0,2006,50000
1,2015,50000
2,2016,50000
3,2017,50000


In [3]:
def construct_vintage_outcomes(
    vintage,
    performance_file,
    origination_reference,
    horizons=(12, 24, 36),
    chunk_size=100_000,
):
    """
    Construct loan-level delinquency outcomes for one vintage.

    A default event is delinquency status 03 or higher,
    or REO acquisition status RA.

    Nondefault outcomes require a complete monthly window.
    Loans with incomplete windows and no observed default
    are classified as censored.
    """

    vintage_reference = (
        origination_reference.loc[
            origination_reference["vintage"].eq(vintage),
            [
                "loan_identifier",
                "first_payment_date",
            ],
        ]
        .copy()
        .set_index("loan_identifier")
    )

    vintage_reference[
        "first_payment_month_index"
    ] = (
        vintage_reference["first_payment_date"].dt.year
        * 12
        + vintage_reference["first_payment_date"].dt.month
    )

    first_payment_lookup = vintage_reference[
        "first_payment_month_index"
    ]

    outcomes = pd.DataFrame(
        index=vintage_reference.index
    )

    outcomes["performance_row_count"] = 0

    for horizon in horizons:
        outcomes[
            f"observed_months_{horizon}m"
        ] = 0

        outcomes[
            f"event_observed_{horizon}m"
        ] = False

    columns_needed = [
        "loan_identifier",
        "monthly_reporting_period",
        "current_loan_delinquency_status",
    ]

    total_performance_rows = 0
    start_time = time.time()

    reader = pd.read_csv(
        performance_file,
        sep="|",
        header=None,
        names=performance_columns,
        usecols=columns_needed,
        dtype="string",
        chunksize=chunk_size,
        low_memory=False,
    )

    for chunk in reader:
        total_performance_rows += len(chunk)

        loan_ids = (
            chunk["loan_identifier"]
            .fillna("")
            .str.strip()
        )

        reporting_period = pd.to_numeric(
            chunk["monthly_reporting_period"],
            errors="coerce",
        )

        reporting_year = (
            reporting_period // 100
        )

        reporting_month = (
            reporting_period % 100
        )

        reporting_month_index = (
            reporting_year * 12
            + reporting_month
        )

        first_payment_month_index = (
            loan_ids.map(first_payment_lookup)
        )

        months_from_first_payment = (
            reporting_month_index
            - first_payment_month_index
        )

        delinquency_status = (
            chunk[
                "current_loan_delinquency_status"
            ]
            .fillna("")
            .str.strip()
            .str.upper()
        )

        numeric_delinquency = pd.to_numeric(
            delinquency_status,
            errors="coerce",
        )

        default_event = (
            numeric_delinquency.ge(3)
            | delinquency_status.eq("RA")
        ).fillna(False)

        # Count all available performance records by loan.
        row_counts = loan_ids.value_counts()

        outcomes.loc[
            row_counts.index,
            "performance_row_count",
        ] += row_counts

        for horizon in horizons:
            within_window = (
                months_from_first_payment.ge(0)
                & months_from_first_payment.lt(horizon)
            ).fillna(False)

            observed_counts = (
                loan_ids.loc[within_window]
                .value_counts()
            )

            outcomes.loc[
                observed_counts.index,
                f"observed_months_{horizon}m",
            ] += observed_counts

            event_ids = loan_ids.loc[
                within_window & default_event
            ].unique()

            outcomes.loc[
                event_ids,
                f"event_observed_{horizon}m",
            ] = True

    # Assign final outcomes and censoring indicators.
    for horizon in horizons:
        event_column = (
            f"event_observed_{horizon}m"
        )

        observed_column = (
            f"observed_months_{horizon}m"
        )

        complete_window = (
            outcomes[observed_column].ge(horizon)
        )

        event_observed = outcomes[event_column]

        outcome = pd.Series(
            pd.NA,
            index=outcomes.index,
            dtype="Int8",
        )

        outcome.loc[
            complete_window & ~event_observed
        ] = 0

        outcome.loc[event_observed] = 1

        outcomes[f"default_{horizon}m"] = outcome

        outcomes[f"eligible_{horizon}m"] = (
            outcome.notna()
        )

        outcomes[f"censored_{horizon}m"] = (
            outcome.isna()
        )

    outcomes.insert(0, "vintage", vintage)

    outcomes = (
        outcomes
        .reset_index()
        .rename(
            columns={
                "index": "loan_identifier"
            }
        )
    )

    elapsed_seconds = round(
        time.time() - start_time,
        1,
    )

    summary_values = {
        "vintage": vintage,
        "performance_rows_processed": (
            total_performance_rows
        ),
        "loan_outcome_rows": len(outcomes),
        "loans_without_performance": int(
            outcomes[
                "performance_row_count"
            ].eq(0).sum()
        ),
    }

    for horizon in horizons:
        eligible = outcomes[
            f"eligible_{horizon}m"
        ]

        defaults = outcomes[
            f"default_{horizon}m"
        ]

        summary_values.update(
            {
                f"eligible_{horizon}m": int(
                    eligible.sum()
                ),
                f"censored_{horizon}m": int(
                    (~eligible).sum()
                ),
                f"defaults_{horizon}m": int(
                    defaults.eq(1).sum()
                ),
                f"default_rate_{horizon}m": round(
                    float(defaults.mean()),
                    6,
                ),
            }
        )

    summary_values[
        "elapsed_seconds"
    ] = elapsed_seconds

    outcome_summary = pd.Series(
        summary_values,
        name="result",
    )

    return outcomes, outcome_summary

In [4]:
outcomes_2017, summary_2017 = (
    construct_vintage_outcomes(
        vintage=2017,
        performance_file=performance_files[2017],
        origination_reference=origination_reference,
        horizons=(12, 24, 36),
        chunk_size=100_000,
    )
)

display(summary_2017)
display(outcomes_2017.head())

vintage                       2.017000e+03
performance_rows_processed    2.800219e+06
loan_outcome_rows             5.000000e+04
loans_without_performance     0.000000e+00
eligible_12m                  4.591300e+04
censored_12m                  4.087000e+03
defaults_12m                  1.910000e+02
default_rate_12m              4.160000e-03
eligible_24m                  4.206400e+04
censored_24m                  7.936000e+03
defaults_24m                  3.970000e+02
default_rate_24m              9.438000e-03
eligible_36m                  3.289500e+04
censored_36m                  1.710500e+04
defaults_36m                  1.671000e+03
default_rate_36m              5.079800e-02
elapsed_seconds               1.420000e+01
Name: result, dtype: float64

,loan_identifier,vintage,performance_row_count,observed_months_12m,event_observed_12m,observed_months_24m,event_observed_24m,observed_months_36m,event_observed_36m,default_12m,eligible_12m,censored_12m,default_24m,eligible_24m,censored_24m,default_36m,eligible_36m,censored_36m
0,F17Q10000002,2017,57,12,False,24,False,36,False,0,True,False,0,True,False,0,True,False
1,F17Q10000017,2017,110,12,False,24,False,36,False,0,True,False,0,True,False,0,True,False
2,F17Q10000064,2017,61,12,False,24,False,36,False,0,True,False,0,True,False,0,True,False
3,F17Q10000065,2017,42,12,False,24,False,36,False,0,True,False,0,True,False,0,True,False
4,F17Q10000073,2017,110,12,False,24,False,36,False,0,True,False,0,True,False,0,True,False


In [5]:
display(summary_2017)

vintage                       2.017000e+03
performance_rows_processed    2.800219e+06
loan_outcome_rows             5.000000e+04
loans_without_performance     0.000000e+00
eligible_12m                  4.591300e+04
censored_12m                  4.087000e+03
defaults_12m                  1.910000e+02
default_rate_12m              4.160000e-03
eligible_24m                  4.206400e+04
censored_24m                  7.936000e+03
defaults_24m                  3.970000e+02
default_rate_24m              9.438000e-03
eligible_36m                  3.289500e+04
censored_36m                  1.710500e+04
defaults_36m                  1.671000e+03
default_rate_36m              5.079800e-02
elapsed_seconds               1.420000e+01
Name: result, dtype: float64

In [6]:
outcomes_by_vintage = {
    2017: outcomes_2017
}

outcome_summary_rows = [
    summary_2017.to_dict()
]

for year in [2006, 2015, 2016]:
    print(f"Constructing {year} outcomes...")

    vintage_outcomes, vintage_summary = (
        construct_vintage_outcomes(
            vintage=year,
            performance_file=performance_files[year],
            origination_reference=origination_reference,
            horizons=(12, 24, 36),
            chunk_size=100_000,
        )
    )

    outcomes_by_vintage[year] = (
        vintage_outcomes
    )

    outcome_summary_rows.append(
        vintage_summary.to_dict()
    )

    print(
        f"{year} complete: "
        f"{len(vintage_outcomes):,} loans"
    )

loan_outcomes = (
    pd.concat(
        [
            outcomes_by_vintage[year]
            for year in vintages
        ],
        ignore_index=True,
    )
)

outcome_summary = (
    pd.DataFrame(outcome_summary_rows)
    .sort_values("vintage")
    .reset_index(drop=True)
)

# Cleaner display using percentages.
outcome_summary_display = outcome_summary[
    [
        "vintage",
        "performance_rows_processed",
        "loan_outcome_rows",
        "loans_without_performance",
        "eligible_12m",
        "censored_12m",
        "defaults_12m",
        "default_rate_12m",
        "eligible_24m",
        "censored_24m",
        "defaults_24m",
        "default_rate_24m",
        "eligible_36m",
        "censored_36m",
        "defaults_36m",
        "default_rate_36m",
        "elapsed_seconds",
    ]
].copy()

for column in [
    "default_rate_12m",
    "default_rate_24m",
    "default_rate_36m",
]:
    outcome_summary_display[column] = (
        outcome_summary_display[column]
        * 100
    ).round(2)

display(outcome_summary_display)

Constructing 2006 outcomes...
2006 complete: 50,000 loans
Constructing 2015 outcomes...
2015 complete: 50,000 loans
Constructing 2016 outcomes...
2016 complete: 50,000 loans


,vintage,performance_rows_processed,loan_outcome_rows,loans_without_performance,eligible_12m,censored_12m,defaults_12m,default_rate_12m,eligible_24m,censored_24m,defaults_24m,default_rate_24m,eligible_36m,censored_36m,defaults_36m,default_rate_36m,elapsed_seconds
0,2006.0,3203499.0,50000.0,0.0,42766.0,7234.0,248.0,0.58,37555.0,12445.0,967.0,2.57,31055.0,18945.0,2521.0,8.12,18.4
1,2015.0,3467166.0,50000.0,0.0,44600.0,5400.0,65.0,0.15,39911.0,10089.0,253.0,0.63,35916.0,14084.0,453.0,1.26,19.2
2,2016.0,3379650.0,50000.0,0.0,45829.0,4171.0,75.0,0.16,42476.0,7524.0,330.0,0.78,38574.0,11426.0,488.0,1.27,19.6
3,2017.0,2800219.0,50000.0,0.0,45913.0,4087.0,191.0,0.42,42064.0,7936.0,397.0,0.94,32895.0,17105.0,1671.0,5.08,14.2


In [7]:
validation_results = {
    "total_outcome_rows": len(
        loan_outcomes
    ),
    "unique_loan_ids": (
        loan_outcomes[
            "loan_identifier"
        ].nunique()
    ),
    "duplicate_loan_ids": int(
        loan_outcomes[
            "loan_identifier"
        ].duplicated().sum()
    ),
    "loans_without_performance": int(
        loan_outcomes[
            "performance_row_count"
        ].eq(0).sum()
    ),
    "performance_rows_reconciled": int(
        loan_outcomes[
            "performance_row_count"
        ].sum()
    ),
}

for horizon in [12, 24, 36]:
    eligible = loan_outcomes[
        f"eligible_{horizon}m"
    ]

    censored = loan_outcomes[
        f"censored_{horizon}m"
    ]

    outcome = loan_outcomes[
        f"default_{horizon}m"
    ]

    event = loan_outcomes[
        f"event_observed_{horizon}m"
    ]

    observed_months = loan_outcomes[
        f"observed_months_{horizon}m"
    ]

    validation_results.update(
        {
            f"eligibility_reconciliation_errors_{horizon}m": int(
                (eligible.eq(censored)).sum()
            ),
            f"invalid_outcome_values_{horizon}m": int(
                (
                    outcome.notna()
                    & ~outcome.isin([0, 1])
                ).sum()
            ),
            f"event_without_default_{horizon}m": int(
                (
                    event
                    & outcome.ne(1).fillna(True)
                ).sum()
            ),
            f"nondefault_without_full_window_{horizon}m": int(
                (
                    outcome.eq(0)
                    & observed_months.lt(horizon)
                ).sum()
            ),
            f"observed_months_above_horizon_{horizon}m": int(
                observed_months.gt(horizon).sum()
            ),
        }
    )

validation_results.update(
    {
        "default_12m_not_preserved_at_24m": int(
            (
                loan_outcomes["default_12m"].eq(1)
                & ~loan_outcomes["default_24m"].eq(1)
            ).sum()
        ),
        "default_24m_not_preserved_at_36m": int(
            (
                loan_outcomes["default_24m"].eq(1)
                & ~loan_outcomes["default_36m"].eq(1)
            ).sum()
        ),
        "observed_months_decrease_12m_to_24m": int(
            (
                loan_outcomes["observed_months_24m"]
                < loan_outcomes["observed_months_12m"]
            ).sum()
        ),
        "observed_months_decrease_24m_to_36m": int(
            (
                loan_outcomes["observed_months_36m"]
                < loan_outcomes["observed_months_24m"]
            ).sum()
        ),
    }
)

outcome_validation = pd.Series(
    validation_results,
    name="result",
)

display(outcome_validation)

total_outcome_rows                         200000
unique_loan_ids                            200000
duplicate_loan_ids                              0
loans_without_performance                       0
performance_rows_reconciled              12850534
eligibility_reconciliation_errors_12m           0
invalid_outcome_values_12m                      0
event_without_default_12m                       0
nondefault_without_full_window_12m              0
observed_months_above_horizon_12m               0
eligibility_reconciliation_errors_24m           0
invalid_outcome_values_24m                      0
event_without_default_24m                       0
nondefault_without_full_window_24m              0
observed_months_above_horizon_24m               0
eligibility_reconciliation_errors_36m           0
invalid_outcome_values_36m                      0
event_without_default_36m                       0
nondefault_without_full_window_36m              0
observed_months_above_horizon_36m               0


In [8]:
# Correct cosmetic data types in the summary.
outcome_summary["vintage"] = (
    outcome_summary["vintage"]
    .astype("int64")
)

integer_summary_columns = [
    column
    for column in outcome_summary.columns
    if (
        column.endswith("_processed")
        or column.endswith("_rows")
        or column.startswith("eligible_")
        or column.startswith("censored_")
        or column.startswith("defaults_")
        or column == "loans_without_performance"
    )
]

for column in integer_summary_columns:
    outcome_summary[column] = (
        outcome_summary[column]
        .astype("int64")
    )


loan_outcomes_path = (
    processed_data_directory
    / "loan_outcomes.parquet"
)

outcome_summary_path = (
    processed_data_directory
    / "outcome_summary.csv"
)


loan_outcomes.to_parquet(
    loan_outcomes_path,
    index=False,
)

outcome_summary.to_csv(
    outcome_summary_path,
    index=False,
)


# Reload the Parquet file for independent validation.
reloaded_outcomes = pd.read_parquet(
    loan_outcomes_path
)

export_validation = pd.Series(
    {
        "export_file_exists": (
            loan_outcomes_path.exists()
        ),
        "summary_file_exists": (
            outcome_summary_path.exists()
        ),
        "exported_rows": len(
            reloaded_outcomes
        ),
        "exported_columns": len(
            reloaded_outcomes.columns
        ),
        "unique_exported_loan_ids": (
            reloaded_outcomes[
                "loan_identifier"
            ].nunique()
        ),
        "duplicate_exported_loan_ids": int(
            reloaded_outcomes[
                "loan_identifier"
            ].duplicated().sum()
        ),
        "performance_rows_reconciled": int(
            reloaded_outcomes[
                "performance_row_count"
            ].sum()
        ),
        "defaults_12m_match": (
            int(
                reloaded_outcomes[
                    "default_12m"
                ].eq(1).sum()
            )
            == int(
                loan_outcomes[
                    "default_12m"
                ].eq(1).sum()
            )
        ),
        "defaults_24m_match": (
            int(
                reloaded_outcomes[
                    "default_24m"
                ].eq(1).sum()
            )
            == int(
                loan_outcomes[
                    "default_24m"
                ].eq(1).sum()
            )
        ),
        "defaults_36m_match": (
            int(
                reloaded_outcomes[
                    "default_36m"
                ].eq(1).sum()
            )
            == int(
                loan_outcomes[
                    "default_36m"
                ].eq(1).sum()
            )
        ),
        "parquet_file_size_mb": round(
            loan_outcomes_path.stat().st_size
            / (1024 ** 2),
            2,
        ),
    },
    name="result",
)

display(export_validation)

export_file_exists                 True
summary_file_exists                True
exported_rows                    200000
exported_columns                     18
unique_exported_loan_ids         200000
duplicate_exported_loan_ids           0
performance_rows_reconciled    12850534
defaults_12m_match                 True
defaults_24m_match                 True
defaults_36m_match                 True
parquet_file_size_mb               1.84
Name: result, dtype: object

## Outcome Construction Results

Loan-level outcomes were successfully constructed for all 200,000 loans
using 12,850,534 validated monthly performance records.

The exported outcome population contains:

- one record per loan;
- 12-, 24-, and 36-month default indicators;
- observation-window eligibility indicators;
- censoring indicators; and
- supporting performance-record counts.

A default event represents delinquency status `03` or higher or REO
acquisition status `RA` during the applicable observation window.

Loans without an observed default require a complete monthly window to
receive a nondefault outcome. Loans that exit before completing the
window are retained but classified as censored.

## Outcome Rates by Vintage

| Vintage | 12-month | 24-month | 36-month |
|---|---:|---:|---:|
| 2006 | 0.58% | 2.57% | 8.12% |
| 2015 | 0.15% | 0.63% | 1.26% |
| 2016 | 0.16% | 0.78% | 1.27% |
| 2017 | 0.42% | 0.94% | 5.08% |

The 2006 vintage exhibits substantially higher early credit risk than
the 2015–2017 vintages.

The 2017 vintage shows a notable increase between the 24- and 36-month
windows. Its calendar timing overlaps the period surrounding 2020, but
calendar-period analysis is required before attributing the increase to
a specific economic event.

## Validation Conclusion

The following controls passed:

- 200,000 outcome records reconciled to 200,000 origination loans;
- all 12,850,534 performance records were accounted for;
- no duplicate loan-level outcome records were created;
- no loans were missing performance data;
- all outcome values were valid;
- eligibility and censoring classifications reconciled;
- nondefault outcomes required complete observation windows;
- default events were preserved across longer horizons; and
- exported default counts matched the in-memory results.

The validated output files are:

- `data/processed/loan_outcomes.parquet`
- `data/processed/outcome_summary.csv`

The outcome population is ready for independent outcome recalculation,
feature engineering, vintage analysis, and subsequent PD modeling.

In [10]:
audit_sample_parts = []

selection_rules = {
    "default": lambda data: (
        data["default_24m"].eq(1)
    ),
    "nondefault": lambda data: (
        data["default_24m"].eq(0)
    ),
    "censored": lambda data: (
        data["default_24m"].isna()
    ),
}

for year in vintages:
    vintage_data = loan_outcomes.loc[
        loan_outcomes["vintage"].eq(year)
    ].copy()

    for group_number, (
        selection_group,
        selection_rule,
    ) in enumerate(
        selection_rules.items(),
        start=1,
    ):
        eligible_sample = vintage_data.loc[
            selection_rule(vintage_data)
        ]

        selected_loans = (
            eligible_sample
            .sample(
                n=10,
                random_state=year + group_number,
            )
            [
                [
                    "loan_identifier",
                    "vintage",
                    "observed_months_24m",
                    "default_24m",
                    "eligible_24m",
                    "censored_24m",
                ]
            ]
            .copy()
        )

        selected_loans[
            "selection_group"
        ] = selection_group

        audit_sample_parts.append(
            selected_loans
        )

outcome_audit_sample = (
    pd.concat(
        audit_sample_parts,
        ignore_index=True,
    )
    .sort_values(
        [
            "vintage",
            "selection_group",
            "loan_identifier",
        ]
    )
    .reset_index(drop=True)
)

sample_design_check = (
    outcome_audit_sample
    .groupby(
        ["vintage", "selection_group"]
    )
    .size()
    .rename("selected_loans")
    .reset_index()
)

print(
    "Total control-sample loans:",
    len(outcome_audit_sample),
)

display(sample_design_check)

Total control-sample loans: 120


,vintage,selection_group,selected_loans
0,2006,censored,10
1,2006,default,10
2,2006,nondefault,10
3,2015,censored,10
4,2015,default,10
5,2015,nondefault,10
6,2016,censored,10
7,2016,default,10
8,2016,nondefault,10
9,2017,censored,10


In [11]:
raw_sample_records = []

columns_needed = [
    "loan_identifier",
    "monthly_reporting_period",
    "current_loan_delinquency_status",
]

for year in vintages:
    print(
        f"Retrieving raw records for "
        f"{year} control sample..."
    )

    sample_ids = set(
        outcome_audit_sample.loc[
            outcome_audit_sample[
                "vintage"
            ].eq(year),
            "loan_identifier",
        ]
    )

    reader = pd.read_csv(
        performance_files[year],
        sep="|",
        header=None,
        names=performance_columns,
        usecols=columns_needed,
        dtype="string",
        chunksize=100_000,
        low_memory=False,
    )

    for chunk in reader:
        loan_ids = (
            chunk["loan_identifier"]
            .fillna("")
            .str.strip()
        )

        selected_mask = loan_ids.isin(
            sample_ids
        )

        if selected_mask.any():
            selected_records = (
                chunk.loc[
                    selected_mask,
                    columns_needed,
                ]
                .copy()
            )

            selected_records.insert(
                0,
                "vintage",
                year,
            )

            raw_sample_records.append(
                selected_records
            )

raw_outcome_audit_records = pd.concat(
    raw_sample_records,
    ignore_index=True,
)

raw_sample_check = pd.Series(
    {
        "selected_loans": len(
            outcome_audit_sample
        ),
        "raw_records_retrieved": len(
            raw_outcome_audit_records
        ),
        "unique_loans_retrieved": (
            raw_outcome_audit_records[
                "loan_identifier"
            ].nunique()
        ),
        "missing_sample_loans": (
            len(outcome_audit_sample)
            - raw_outcome_audit_records[
                "loan_identifier"
            ].nunique()
        ),
    },
    name="result",
)

display(raw_sample_check)

Retrieving raw records for 2006 control sample...
Retrieving raw records for 2015 control sample...
Retrieving raw records for 2016 control sample...
Retrieving raw records for 2017 control sample...


selected_loans             120
raw_records_retrieved     6791
unique_loans_retrieved     120
missing_sample_loans         0
Name: result, dtype: int64

In [12]:
manual_recalculation = (
    raw_outcome_audit_records.merge(
        origination_reference[
            [
                "loan_identifier",
                "first_payment_date",
            ]
        ],
        on="loan_identifier",
        how="left",
        validate="many_to_one",
    )
)

reporting_period = pd.to_numeric(
    manual_recalculation[
        "monthly_reporting_period"
    ],
    errors="coerce",
)

reporting_month_index = (
    (reporting_period // 100) * 12
    + (reporting_period % 100)
)

first_payment_month_index = (
    manual_recalculation[
        "first_payment_date"
    ].dt.year
    * 12
    + manual_recalculation[
        "first_payment_date"
    ].dt.month
)

manual_recalculation[
    "months_from_first_payment"
] = (
    reporting_month_index
    - first_payment_month_index
)

manual_recalculation[
    "within_24m_window"
] = (
    manual_recalculation[
        "months_from_first_payment"
    ].ge(0)
    & manual_recalculation[
        "months_from_first_payment"
    ].lt(24)
)

delinquency_status = (
    manual_recalculation[
        "current_loan_delinquency_status"
    ]
    .fillna("")
    .str.strip()
    .str.upper()
)

numeric_delinquency = pd.to_numeric(
    delinquency_status,
    errors="coerce",
)

manual_recalculation[
    "default_event_in_24m"
] = (
    manual_recalculation[
        "within_24m_window"
    ]
    & (
        numeric_delinquency.ge(3)
        | delinquency_status.eq("RA")
    )
)

manual_loan_results = (
    manual_recalculation
    .groupby(
        ["vintage", "loan_identifier"],
        as_index=False,
    )
    .agg(
        raw_record_count=(
            "monthly_reporting_period",
            "size",
        ),
        manually_observed_months_24m=(
            "within_24m_window",
            "sum",
        ),
        manually_observed_default_24m=(
            "default_event_in_24m",
            "max",
        ),
    )
)

manual_outcome = pd.Series(
    pd.NA,
    index=manual_loan_results.index,
    dtype="Int8",
)

manual_complete_window = (
    manual_loan_results[
        "manually_observed_months_24m"
    ].ge(24)
)

manual_default_event = (
    manual_loan_results[
        "manually_observed_default_24m"
    ]
)

manual_outcome.loc[
    manual_complete_window
    & ~manual_default_event
] = 0

manual_outcome.loc[
    manual_default_event
] = 1

manual_loan_results[
    "manually_calculated_default_24m"
] = manual_outcome

manual_loan_results[
    "manually_calculated_eligible_24m"
] = manual_outcome.notna()

manual_loan_results[
    "manually_calculated_censored_24m"
] = manual_outcome.isna()


outcome_audit_comparison = (
    outcome_audit_sample.merge(
        manual_loan_results,
        on=[
            "vintage",
            "loan_identifier",
        ],
        how="left",
        validate="one_to_one",
    )
)

outcome_audit_comparison[
    "observed_months_match"
] = (
    outcome_audit_comparison[
        "observed_months_24m"
    ]
    .eq(
        outcome_audit_comparison[
            "manually_observed_months_24m"
        ]
    )
)

outcome_audit_comparison[
    "outcome_match"
] = (
    outcome_audit_comparison[
        "default_24m"
    ]
    .fillna(-1)
    .astype("int8")
    .eq(
        outcome_audit_comparison[
            "manually_calculated_default_24m"
        ]
        .fillna(-1)
        .astype("int8")
    )
)

outcome_audit_comparison[
    "eligibility_match"
] = (
    outcome_audit_comparison[
        "eligible_24m"
    ]
    .eq(
        outcome_audit_comparison[
            "manually_calculated_eligible_24m"
        ]
    )
)

outcome_audit_comparison[
    "censoring_match"
] = (
    outcome_audit_comparison[
        "censored_24m"
    ]
    .eq(
        outcome_audit_comparison[
            "manually_calculated_censored_24m"
        ]
    )
)


independent_recalculation_summary = pd.Series(
    {
        "sample_loans_tested": len(
            outcome_audit_comparison
        ),
        "observed_months_matches": int(
            outcome_audit_comparison[
                "observed_months_match"
            ].sum()
        ),
        "outcome_matches": int(
            outcome_audit_comparison[
                "outcome_match"
            ].sum()
        ),
        "eligibility_matches": int(
            outcome_audit_comparison[
                "eligibility_match"
            ].sum()
        ),
        "censoring_matches": int(
            outcome_audit_comparison[
                "censoring_match"
            ].sum()
        ),
        "total_mismatches": int(
            (
                ~outcome_audit_comparison[
                    [
                        "observed_months_match",
                        "outcome_match",
                        "eligibility_match",
                        "censoring_match",
                    ]
                ].all(axis=1)
            ).sum()
        ),
        "all_independent_checks_passed": bool(
            outcome_audit_comparison[
                [
                    "observed_months_match",
                    "outcome_match",
                    "eligibility_match",
                    "censoring_match",
                ]
            ].all(axis=None)
        ),
    },
    name="result",
)

display(independent_recalculation_summary)

sample_loans_tested               120
observed_months_matches           120
outcome_matches                   120
eligibility_matches               120
censoring_matches                 120
total_mismatches                    0
all_independent_checks_passed    True
Name: result, dtype: object

## Independent Outcome Recalculation

A control-focused sample of 120 loans was selected for independent
24-month outcome recalculation.

The sample included, for each of the four vintages:

- 10 loans classified as default;
- 10 loans classified as nondefault; and
- 10 loans classified as censored.

The sample was intentionally stratified across outcome classifications
and was used for control validation rather than population-rate
estimation.

A total of 6,791 monthly records for the selected loans were retrieved
directly from the raw performance files. The 24-month observation
window, default event, eligibility status, and censoring status were
then recalculated separately from the exported outcome table.

## Recalculation Results

| Validation measure | Result |
|---|---:|
| Loans tested | 120 |
| Observation-window matches | 120 |
| Outcome matches | 120 |
| Eligibility matches | 120 |
| Censoring matches | 120 |
| Total mismatches | 0 |

All independently recalculated results matched the constructed
loan-level outcome table.

## EIT Conclusion

Outcome-window eligibility testing and independent outcome
recalculation passed without exception.

The outcome-construction methodology correctly:

- identifies 90-plus-day delinquency and REO events;
- applies the selected 12-, 24-, and 36-month windows;
- requires complete observation for nondefault outcomes;
- distinguishes incomplete observations as censored; and
- maintains one validated outcome record per loan.

The 24-month outcome is approved for subsequent feature engineering,
vintage analysis, and probability-of-default modeling.